In [0]:
df = spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv"
                    ,header = True
                    ,inferSchema = True
                    ,sep = "|"
                    ,quote ="'"
                    
                    )
df.display()

##Handling missing Records

In [0]:
df.na.drop(subset=["id","name"]).display()

##DRop column when both colum is null

In [0]:
from pyspark.sql.functions import col

df.filter((col("name").isNotNull()) & (col("company").isNotNull())).display()

##Fill null file with default value

In [0]:
df.na.fill("Null_by_default").na.fill(-1).display()

##Fill Null value with different data types

In [0]:
COLUMN_WITH_DEFAULT= {
    "id":-1
    ,"name":'Anonymous'
    ,"exp":0
    ,"doj":"1970-01-01"
}

df.na.fill(COLUMN_WITH_DEFAULT).display()

##Fill null value with avg experiences

In [0]:
from pyspark.sql.functions import avg
avg_exp = df.select(avg("exp")).collect()[0][0]
COLUMN_WITH_DEFAULT= {
    "id":-1
    ,"name":'Anonymous'
    ,"exp":int(round(avg_exp,2))
    ,"doj":"1970-01-01"
}
df.na.fill(COLUMN_WITH_DEFAULT).display()


##Fill categorical data


In [0]:
from pyspark.sql.functions import col
df.groupBy("gen").count().orderBy(col("count").desc()).display()
defualt_gender=df.groupBy("gen").count().orderBy(col("count").desc()).first()[0]
print(defualt_gender)
COLUMN_WITH_DEFAULT= {
    "id":-1
    ,"name":'Anonymous'
    ,"exp":int(round(avg_exp,2))
    ,"gen":defualt_gender
}
df.na.fill(COLUMN_WITH_DEFAULT).display()



In [0]:
from pyspark.sql.functions import col,avg,when

avg_exp_1 = df.filter(col("exp")>0).agg(avg("exp")).collect()[0][0]
print(avg_exp_1)
df.withColumn("exp",when((col("exp").isNull())|(col("exp") < 0),avg_exp_1).otherwise(col("exp"))).display()